In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!ls /content/drive/MyDrive/agribot_v4/data

 Crop_recommendation.csv
'Crop Recommendation using Soil Properties and Weather Prediction.csv'
 crop_yield.csv
'Crop Yiled with Soil and Weather.csv'
 data_core.csv
'Fertilizer Prediction.csv'
 state_soil_data.csv
 state_weather_data_1997_2020.csv


In [1]:
%cd /content/drive/MyDrive/agribot_v4
!pip install -r requirements.txt

/content/drive/MyDrive/agribot_v4


In [1]:
# ── STEP 3: Install all dependencies ───────────────────────────
!pip install xgboost ultralytics
import sklearn, transformers, torch
print("sklearn:", sklearn.__version__)
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())
print("Ready!")

sklearn: 1.6.1
torch: 2.11.0+cu128 | GPU: True
Ready!


In [2]:
# ── STEP 4A: Train CORE models (RF crop + RF fertilizer + XGBoost) ──
%cd /content/drive/MyDrive/agribot_v4
!python training/train_crop_model.py
!python training/train_fertilizer_model.py
!python training/train_xgboost_crop.py

/content/drive/MyDrive/agribot_v4
  AgriBot — Crop Model Training  (v4)
[CROP] Loading Dataset 1: Kaggle Crop Recommendation (2200 rows) ...
       → 2200 rows | 22 crops: ['apple', 'banana', 'blackgram', 'chickpea', 'coconut', 'coffee', 'cotton', 'grapes', 'jute', 'kidneybeans', 'lentil', 'maize', 'mango', 'mothbeans', 'mungbean', 'muskmelon', 'orange', 'papaya', 'pigeonpeas', 'pomegranate', 'rice', 'watermelon']
[CROP] Training RandomForest ...
[CROP] Test Accuracy: 0.9955
              precision    recall  f1-score   support

       apple       1.00      1.00      1.00        20
      banana       1.00      1.00      1.00        20
   blackgram       1.00      0.95      0.97        20
    chickpea       1.00      1.00      1.00        20
     coconut       1.00      1.00      1.00        20
      coffee       1.00      1.00      1.00        20
      cotton       1.00      1.00      1.00        20
      grapes       1.00      1.00      1.00        20
        jute       0.95      1.00

In [3]:
# Train NLP models (DistilBERT + Full BERT)
!python /content/drive/MyDrive/agribot_v4/training/train_intent_classifier.py
!python /content/drive/MyDrive/agribot_v4/training/train_bert_intent.py

[INFO] Device: cuda
[INFO] Total samples: 900
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 128kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.01MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 2.55MB/s]
config.json: 100% 483/483 [00:00<00:00, 1.88MB/s]
model.safetensors: 100% 268M/268M [00:02<00:00, 91.0MB/s]
Loading weights: 100% 100/100 [00:00<00:00, 1718.36it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ign

In [4]:
!python /content/drive/MyDrive/agribot_v4/training/train_disease_efficientnet.py
!python /content/drive/MyDrive/agribot_v4/training/train_disease_yolo.py

  EfficientNet Plant Disease Classifier

[WARN] PlantVillage dataset not found at: 'PlantVillage'
       To download, run one of:
       → kaggle datasets download abdallahalidev/plantvillage-dataset
       → !pip install tensorflow-datasets
         import tensorflow_datasets as tfds
         ds = tfds.load('plant_village', split='train[:80%]')

[INFO] Generating SIMULATED training run for demonstration...
[INFO] Saving representative metrics from published benchmarks ...
[DONE] Simulated meta saved. Accuracy=98.43% (PlantVillage benchmark)
  YOLOv8 Plant Disease Detection

[WARN] YOLO dataset YAML not found at: 'plant_disease.yaml'
       To get the dataset:
       1. Sign up at https://universe.roboflow.com
       2. Download 'PlantDoc' dataset in YOLOv8 format
       3. Set: export YOLO_DATASET_YAML=/path/to/plant_disease.yaml
       4. Run this script again

[INFO] Saving benchmark metrics for comparison ...

  DISEASE MODEL COMPARISON — EfficientNet vs YOLO
Metric                

In [5]:
import os, json
model_dir = "/content/drive/MyDrive/agribot_v4/models"
expected = ["crop_model.pkl", "fertilizer_model.pkl", "xgb_crop_model.pkl",
            "crop_meta.json", "fertilizer_meta.json", "xgb_crop_meta.json",
            "state_profiles.json", "crop_yield_stats.json",
            "yield_model.pkl", "yield_scaler.pkl"]
print("Core models:")
for f in expected:
    path = os.path.join(model_dir, f)
    exists = "✅" if os.path.exists(path) else "❌"
    print(f"  {exists} {f}")

intent_dir = os.path.join(model_dir, "intent_model")
bert_dir   = os.path.join(model_dir, "bert_intent_model")
print(f"\nDistilBERT: {'✅' if os.path.exists(intent_dir) else '❌'}")
print(f"Full BERT:   {'✅' if os.path.exists(bert_dir) else '❌'}")

# Show comparison
comp_path = os.path.join(model_dir, "comparison_crop_rf_vs_xgb.json")
if os.path.exists(comp_path):
    with open(comp_path) as f: c = json.load(f)
    print(f"\nCrop RF vs XGBoost winner: {c['winner']}")
    for row in c['slide_table'][1:]:
        print(f"  {row[0]:<22} RF:{row[1]:>8}  XGB:{row[2]:>8}  {row[3]}")

Core models:
  ✅ crop_model.pkl
  ✅ fertilizer_model.pkl
  ✅ xgb_crop_model.pkl
  ✅ crop_meta.json
  ✅ fertilizer_meta.json
  ✅ xgb_crop_meta.json
  ✅ state_profiles.json
  ✅ crop_yield_stats.json
  ✅ yield_model.pkl
  ✅ yield_scaler.pkl

DistilBERT: ✅
Full BERT:   ✅

Crop RF vs XGBoost winner: RandomForest
  Accuracy               RF:  0.9955  XGB:  0.9932  RF
  Precision (macro)      RF:  0.9957  XGB:  0.9935  —
  Recall (macro)         RF:  0.9955  XGB:  0.9932  —
  F1-Score (macro)       RF:  0.9955  XGB:  0.9931  RF
  CV Mean (5-fold)       RF:  0.9945  XGB:  0.9941  —
  Train Time (s)         RF:    1.23  XGB:    9.27  —


In [6]:
# Run evaluation on 700 farmer queries
!python /content/drive/MyDrive/agribot_v4/evaluation/evaluate_all.py
!cat /content/drive/MyDrive/agribot_v4/evaluation/test_results.csv | head -20

  AgriBot v4 — Complete Model Evaluation

  1. CROP RECOMMENDATION — RandomForest vs XGBoost
Metric                  RandomForest (Baseline) XGBoost     Winner        
--------------------------------------------------------------------------
Accuracy                0.9955                  0.9932      RF            
Precision (macro)       0.9957                  0.9935      —             
Recall (macro)          0.9955                  0.9932      —             
F1-Score (macro)        0.9955                  0.9931      RF            
CV Mean (5-fold)        0.9945                  0.9941      —             
Train Time (s)          1.23                    9.27        —             

🏆 Winner: RandomForest

  2. NLP INTENT CLASSIFICATION — DistilBERT vs Full BERT
Metric                  DistilBERT    BERT (Full)   Winner          
--------------------------------------------------------------------
Accuracy                1.0000        1.0000        DistilBERT      
Precision (macro) 

In [7]:
# FastAPI server ─────────────────────────────────
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token("3CyrYyPMfuQUvDwAQPujuBeb8Tu_2EafA4pUE9Rhzsxe3BF9G")

import subprocess, time
server = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/drive/MyDrive/agribot_v4")
time.sleep(6)
tunnel = ngrok.connect(8000, "http")
PUBLIC_URL = tunnel.public_url
print(f"\nAgriBot v4 API: {PUBLIC_URL}")
print(f"Swagger Docs:   {PUBLIC_URL}/docs")
print(f"Chatbot UI:     {PUBLIC_URL}/ui")


AgriBot v4 API: https://justly-dispersal-macaw.ngrok-free.dev
Swagger Docs:   https://justly-dispersal-macaw.ngrok-free.dev/docs
Chatbot UI:     https://justly-dispersal-macaw.ngrok-free.dev/ui


In [9]:
# Test all endpoints
import requests, json

BASE = PUBLIC_URL

tests = [
    # Dosage-specific (reasoning engine)
    ("fertilizer_dosage", "/chat", {"message": "How much urea for rice per acre?"}),
    # Crop suitability with conditions (reasoning engine)
    ("suitability_hot",   "/chat", {"message": "30°C and 75% humidity — can I grow maize?"}),
    ("suitability_cold",  "/chat", {"message": "It is 12°C now. Should I plant rice?"}),
    # Natural language
    ("crop_sandy",        "/chat", {"message": "Can I grow potatoes in sandy soil?"}),
    ("fertilizer_flower", "/chat", {"message": "Which fertilizer is best for flowering plants?"}),
    ("disease_rice",      "/chat", {"message": "My rice leaves are turning yellow!"}),
    # State-based (uses 1997-2020 real data)
    ("state_punjab",      "/chat", {"message": "I am from Punjab. What crop should I grow?"}),
    # Direct reasoning API
    ("dosage_api",   "/reasoning/dosage",      {"fertilizer":"urea","crop":"wheat"}),
    ("suitability_api","/reasoning/suitability",{"crop":"maize","temperature":30,"humidity":75}),
    # Model comparison endpoints
    ("rf_vs_xgb",    "/compare/crop",  None),
    ("disease_comp", "/compare/disease", None),
]

print("=" * 65)
for name, endpoint, body in tests:
    try:
        if body:
            r = requests.post(f"{BASE}{endpoint}", json=body, timeout=15)
        else:
            r = requests.get(f"{BASE}{endpoint}", timeout=10)
        d = r.json()
        if endpoint == "/chat":
            print(f"\n[{name}] {body['message'][:55]}...")
            print(f"  Intent: {d.get('intent','')} ({d.get('confidence',0)*100:.1f}%)")
            print(f"  Response: {d.get('response','')[:100]}...")
        elif endpoint == "/compare/crop":
            print(f"\n[{name}] Winner: {d.get('winner','?')}")
        else:
            resp = str(d)[:100]
            print(f"\n[{name}] {endpoint}: {resp}...")
    except Exception as e:
        print(f"\n[{name}] ERROR: {e}")
print("\n" + "=" * 65)


[fertilizer_dosage] How much urea for rice per acre?...
  Intent: fertilizer_advice (99.4%)
  Response: ⚖️ **Urea Dosage for Rice**

**Recommended dose:** 80–100 kg/acre
**Application timing:** split: 1/3...

[suitability_hot] 30°C and 75% humidity — can I grow maize?...
  Intent: crop_recommendation (97.4%)
  Response: 🌱 **Crop Suitability Analysis: Maize**

**✅ Conditions that are suitable:**
   ✅ Temperature 30.0°C ...

[suitability_cold] It is 12°C now. Should I plant rice?...
  Intent: weather_planting (99.2%)
  Response: 🌱 **Crop Suitability Analysis: Rice**

**⚠️ Conditions that are problematic:**
❄️ **Temperature 12.0...

[crop_sandy] Can I grow potatoes in sandy soil?...
  Intent: crop_recommendation (99.5%)
  Response: 🌾 **Crop Recommendations**

**1. Pigeonpeas** (18.5% match)
   → Drought resistant legume. Grows in ...

[fertilizer_flower] Which fertilizer is best for flowering plants?...
  Intent: fertilizer_advice (99.5%)
  Response: 🧪 **Fertilizer Advice**

**Based on y

In [10]:
# Configure live weather
import os
os.environ["OPENWEATHER_API_KEY"] = "6f8ca7e2e01aabd07756f1ff9f3b7ce1"

r = requests.get(f"{PUBLIC_URL}/weather/current?city=Delhi")
print("Delhi weather:", r.json())

r = requests.get(f"{PUBLIC_URL}/weather/planting-advice?city=Mumbai&crop=rice")
print("Planting advice:", r.json())

Delhi weather: {'error': 'Weather data unavailable. Set OPENWEATHER_API_KEY env variable.', 'setup': 'Get free key at openweathermap.org/api'}
Planting advice: {'city': 'Mumbai', 'crop': 'rice', 'current_weather': None, 'forecast': None, 'planting_window': 'unknown', 'recommendation': 'Live weather data unavailable for Mumbai. Please check local weather and use the general planting guide.', 'api_status': 'unavailable'}


In [11]:
# ── STEP 10: Final comparison summary ────────────────────────────
r = requests.get(f"{PUBLIC_URL}/compare/all")
summary = r.json()
print(json.dumps(summary, indent=2)[:3000])

{
  "project": "AgriBot v4 \u2014 Agricultural Advisory Chatbot",
  "evaluation_date": "2026-06-02",
  "task_1_recommendation": {
    "title": "Crop Recommendation \u2014 RF vs XGBoost",
    "winner": "RandomForest",
    "RandomForest": {
      "accuracy": 0.9955,
      "f1_macro": 0.9955,
      "note": "Baseline production model"
    },
    "XGBoost": {
      "accuracy": 0.9932,
      "f1_macro": 0.9931,
      "cv_mean": 0.9941,
      "note": "Comparison model \u2014 gradient boosting"
    }
  },
  "task_2_nlp_intent": {
    "title": "NLP Intent Classification \u2014 DistilBERT vs BERT",
    "winner": "BERT",
    "test_set_accuracy_distilbert": 0.98,
    "DistilBERT": {
      "accuracy": 1.0,
      "precision_macro": 1.0,
      "recall_macro": 1.0,
      "f1_macro": 1.0,
      "precision_weighted": 1.0,
      "recall_weighted": 1.0,
      "f1_weighted": 1.0,
      "confusion_matrix": [
        [
          23,
          0,
          0,
          0
        ],
        [
          0,
    

In [12]:
print(PUBLIC_URL)

https://justly-dispersal-macaw.ngrok-free.dev


In [13]:
!curl http://localhost:8000

{"status":"ok","service":"AgriBot","version":"2.0.0"}

In [14]:
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [['how', 'much', 'urea', 'should', 'i', 'use', 'for', 'rice', 'per', 'acre']]

candidate = ['how', 'much', 'urea', 'for', 'rice', 'per', 'acre']

score = sentence_bleu(reference, candidate)

smoothie = SmoothingFunction().method1
smoothed_score = sentence_bleu(reference, candidate, smoothing_function=smoothie)

print(f"Standard BLEU Score: {score:.4f}")
print(f"Smoothed BLEU Score: {smoothed_score:.4f}")

Standard BLEU Score: 0.3873
Smoothed BLEU Score: 0.3873
